# Quick Start

## How to build a GPU-accelerated FDM
### 1. Download your data
eg. wget https://dept.aem.umn.edu/~./faculty/balas/darpa_sec/software/F16Simulation.tar.gz
Download your aerodata and rename the data file as 'data'

### 2. Train your MLP model
example code, full code see `./train_model/train_model.py  `

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import csv
from train_model.hifi_F16_AeroData import hifi_F16

# 设置设备，优先使用GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_list):
        super(MLP, self).__init__()
        layers = []
        last_dim = in_dim
        for hidden_dim in hidden_list:
            layers.append(nn.Linear(last_dim, hidden_dim))
            layers.append(nn.ReLU())
            last_dim = hidden_dim
        layers.append(nn.Linear(last_dim, out_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = x.float()  # 确保输入为float32
        return self.network(x)


class MyDataset(Dataset):
    def __init__(self, inputs, outputs, transform=None):
        super(MyDataset, self).__init__()
        self.inputs = inputs
        self.outputs = outputs
        self.transform = transform

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        input_sample = self.inputs[idx]
        output_sample = self.outputs[idx]
        if self.transform:
            input_sample = self.transform(input_sample)
            output_sample = self.transform(output_sample)
        return input_sample, output_sample


def safe_read_dat(dat_name):
    """
    安全读取数据文件，返回numpy数组。
    """
    path = os.path.join('./data', dat_name)
    try:
        data = np.loadtxt(path)
        return data
    except OSError:
        print(f"Cannot find file {path} in current directory")
        return np.array([])


def normalize(X, eps=1e-8):
    """
    对张量进行标准化。
    """
    return (X - torch.mean(X)) / (torch.std(X) + eps)


def tensor_to_numpy(tensor):
    """
    将张量转换为numpy数组。
    """
    return tensor.detach().cpu().numpy()


def train_model(train_X, train_Y, file_name, input_dim, output_dim):
    """
    训练单个模型，并保存最佳模型及训练结果。
    """
    # 划分训练集和测试集
    X_train, X_test, y_train, y_test = train_test_split(train_X, train_Y, test_size=0.2, shuffle=True)

    # 创建数据集和数据加载器
    train_dataset = MyDataset(X_train, y_train)
    test_dataset = MyDataset(X_test, y_test)
    BATCH_SIZE = 32
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    # 初始化模型、损失函数和优化器
    hidden_layers = [20, 10]
    model = MLP(input_dim, output_dim, hidden_layers).to(device)
    criterion = nn.L1Loss()
    optimizer = optim.SGD(model.parameters(), lr=0.006, momentum=0.9, weight_decay=5e-4)

    # 使用学习率调度器
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[500, 750, 900], gamma=0.1)

    num_epochs = 1000
    train_loss_history = []
    train_r2_history = []
    test_loss_history = []
    test_r2_history = []
    best_test_r2 = 0.97
    best_model_state = None
    best_test_loss = float('inf')

    # 确保保存目录存在
    os.makedirs("./model", exist_ok=True)
    os.makedirs("./train_result", exist_ok=True)

    for epoch in range(num_epochs):
        # 训练阶段
        model.train()
        train_loss = 0.0
        train_preds = []
        train_targets = []

        for inputs, targets in train_loader:
            inputs = inputs.to(device).float()
            targets = targets.to(device).float()

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            train_preds.append(outputs)
            train_targets.append(targets)

        scheduler.step()  # 更新学习率

        # 计算训练损失和R2
        train_loss /= len(train_loader.dataset)
        train_preds = torch.cat(train_preds).cpu().detach().numpy()
        train_targets = torch.cat(train_targets).cpu().detach().numpy()
        train_r2 = r2_score(train_targets, train_preds)
        train_loss_history.append(train_loss)
        train_r2_history.append(train_r2)

        # 测试阶段
        model.eval()
        test_loss = 0.0
        test_preds = []
        test_targets = []

        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs = inputs.to(device).float()
                targets = targets.to(device).float()

                outputs = model(inputs)
                loss = criterion(outputs, targets)

                test_loss += loss.item() * inputs.size(0)
                test_preds.append(outputs)
                test_targets.append(targets)

        # 计算测试损失和R2
        test_loss /= len(test_loader.dataset)
        test_preds = torch.cat(test_preds).cpu().detach().numpy()
        test_targets = torch.cat(test_targets).cpu().detach().numpy()
        test_r2 = r2_score(test_targets, test_preds)
        test_loss_history.append(test_loss)
        test_r2_history.append(test_r2)

        # 输出当前epoch的结果
        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'  Train Loss: {train_loss:.4f}, Train R2: {train_r2:.4f}')
        print(f'  Test Loss: {test_loss:.4f}, Test R2: {test_r2:.4f}')
        print(f'  Best Test R2 so far: {best_test_r2:.4f}')

        # 检查是否为最佳模型
        if test_r2 > best_test_r2:
            best_test_r2 = test_r2
            best_test_loss = test_loss
            best_model_state = model.state_dict()
            # 保存最佳模型
            model_path = os.path.join("./model", f"{file_name}-{best_test_r2:.4f}-{best_test_loss:.4f}.pth")
            torch.save(best_model_state, model_path)
            print(f'  => Saved new best model to {model_path}')
        if test_r2 > 0.99:
            # 保存训练结果到CSV
            results_path = os.path.join("./train_result", f"{file_name}_result_loss.csv")
            with open(results_path, 'w', newline='') as csvfile:
                csv_writer = csv.writer(csvfile)
                csv_writer.writerow(["train_loss", "test_loss", "test_r2"])
                for tl, tsl, tr2 in zip(train_loss_history, test_loss_history, test_r2_history):
                    csv_writer.writerow([tl, tsl, tr2])
            print(f'Training complete. Best Test R2: {best_test_r2:.4f}, Best Test Loss: {best_test_loss:.4f}')
            return

    # 保存训练结果到CSV
    results_path = os.path.join("./train_result", f"{file_name}_result_loss.csv")
    with open(results_path, 'w', newline='') as csvfile:
        csv_writer = csv.writer(csvfile)
        csv_writer.writerow(["train_loss", "test_loss", "test_r2"])
        for tl, tsl, tr2 in zip(train_loss_history, test_loss_history, test_r2_history):
            csv_writer.writerow([tl, tsl, tr2])

    print(f'Training complete. Best Test R2: {best_test_r2:.4f}, Best Test Loss: {best_test_loss:.4f}')
    return


def main():
    # 生成训练数据
    hifi = hifi_F16()

    # 读取数据文件
    ALPHA1 = safe_read_dat('ALPHA1.dat')
    ALPHA2 = safe_read_dat('ALPHA2.dat')
    BETA1 = safe_read_dat('BETA1.dat')
    DH1 = safe_read_dat('DH1.dat')
    DH2 = safe_read_dat('DH2.dat')

    # 检查数据是否成功加载
    if len(ALPHA1) == 0 or len(ALPHA2) == 0 or len(BETA1) == 0 or len(DH1) == 0 or len(DH2) == 0:
        print("Error: One or more data files could not be loaded.")
        return

    # 生成网格数据
    raw_alpha1 = np.linspace(ALPHA1[0], ALPHA1[-1], 30)
    raw_beta = np.linspace(BETA1[0], BETA1[-1], 30)
    raw_el = np.linspace(DH1[0], DH1[-1], 30)

    alpha = np.tile(raw_alpha1.reshape(-1, 1), raw_beta.shape[0] * raw_el.shape[0]).reshape(-1)
    beta = np.tile(raw_beta.reshape(-1, 1), raw_el.shape[0]).reshape(-1)
    beta = np.tile(beta, raw_alpha1.shape[0])
    el = np.tile(raw_el, raw_alpha1.shape[0] * raw_beta.shape[0])

    # 转换为PyTorch张量
    alpha = torch.tensor(alpha, device=device, dtype=torch.float32, requires_grad=False)
    beta = torch.tensor(beta, device=device, dtype=torch.float32, requires_grad=False)
    el = torch.tensor(el, device=device, dtype=torch.float32, requires_grad=False)

    # 计算目标变量
    Cx = hifi._Cx(alpha, beta, el)
    Cz = hifi._Cz(alpha, beta, el)
    Cm = hifi._Cm(alpha, beta, el)
    Cy = hifi._Cy(alpha, beta)
    Cn = hifi._Cn(alpha, beta, el)
    Cl = hifi._Cl(alpha, beta, el)

    # 记录均值和标准差
    mean_std_path = "./mean_std.csv"
    os.makedirs(os.path.dirname(mean_std_path), exist_ok=True)
    with open(mean_std_path, 'w', newline='') as csvfile:
        csv_writer = csv.writer(csvfile)
        csv_writer.writerow(["name", "alpha_mean", "alpha_std", "beta_mean", "beta_std", "el_mean", "el_std", "mean", "std"])
        for name, C in zip(["Cx", "Cz", "Cm", "Cn", "Cl"], [Cx, Cz, Cm, Cn, Cl]):
            csv_writer.writerow([
                name,
                tensor_to_numpy(torch.mean(alpha)),
                tensor_to_numpy(torch.std(alpha)),
                tensor_to_numpy(torch.mean(beta)),
                tensor_to_numpy(torch.std(beta)),
                tensor_to_numpy(torch.mean(el)),
                tensor_to_numpy(torch.std(el)),
                tensor_to_numpy(torch.mean(C)),
                tensor_to_numpy(torch.std(C))
            ])

    # 标准化数据
    alpha = normalize(alpha)
    beta = normalize(beta)
    el = normalize(el)
    Cx = normalize(Cx)
    Cz = normalize(Cz)
    Cm = normalize(Cm)
    Cn = normalize(Cn)
    Cl = normalize(Cl)

    # 准备训练输入
    train_X = torch.stack((alpha, beta, el), dim=1)
    
    # 训练不同的目标变量
    for target_name, train_Y in zip(["Cx", "Cz", "Cm", "Cn", "Cl"], [Cx, Cz, Cm, Cn, Cl]):
        train_Y = train_Y.view(-1, 1)  # 确保输出维度为 (N, 1)
        print(f'Start training model for {target_name}...')
        train_model(train_X, train_Y, file_name=target_name, input_dim=3, output_dim=1)
        print(f'Finished training model for {target_name}.\n')


if __name__ == "__main__":
    main()

KeyboardInterrupt: 

### 3. Build your FDM
Build your GPU-accelerated FDM

In [1]:
import torch
import torch.nn as nn
import pandas as pd
import os


HIFI_GLOBAL_TXT_CONTENT = {}
# 设置设备，优先使用GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_list):
        super().__init__()
        layers = []
        lastv = in_dim
        for hidden in hidden_list:
            layers.append(nn.Linear(lastv, hidden))
            layers.append(nn.ReLU())
            lastv = hidden
        layers.append(nn.Linear(lastv, out_dim))
        self.layers = nn.Sequential(*layers)
        self.out_dim = out_dim

    def forward(self, x):
        x = x.to(torch.float32)
        ret = self.layers(x)
        ret = ret.reshape(-1)
        return ret
    

def normalize(X, mean, std):
    return (X - mean) / std


def unnormalize(X, mean, std):
    return X * std + mean


class hifi_F16():
    def __init__(self, device=device):
        super().__init__()
        self.data = pd.read_csv('./model/mean_std.csv')
        self.Cx_model = MLP(3, 1, [20, 10]).to(device=device)
        self.Cx_model.load_state_dict(torch.load('./model/Cx.pth', map_location=device))
        self.Cz_model = MLP(3, 1, [20, 10]).to(device=device)
        self.Cz_model.load_state_dict(torch.load('./model/Cz.pth', map_location=device))
        self.Cm_model = MLP(3, 1, [20, 10]).to(device=device)
        self.Cm_model.load_state_dict(torch.load('./model/Cm.pth', map_location=device))
        self.Cy_model = MLP(2, 1, [20, 10]).to(device=device)
        self.Cy_model.load_state_dict(torch.load('./model/Cy.pth', map_location=device))
        self.Cn_model = MLP(3, 1, [20, 10]).to(device=device)
        self.Cn_model.load_state_dict(torch.load('./model/Cn.pth', map_location=device))
        self.Cl_model = MLP(3, 1, [20, 10]).to(device=device)
        self.Cl_model.load_state_dict(torch.load('./model/Cl.pth', map_location=device))
        self.Cxq_model = MLP(1, 1, [20, 10]).to(device=device)
        self.Cxq_model.load_state_dict(torch.load('./model/Cxq.pth', map_location=device))
        self.Czq_model = MLP(1, 1, [20, 10]).to(device=device)
        self.Czq_model.load_state_dict(torch.load('./model/Czq.pth', map_location=device))
        self.Cmq_model = MLP(1, 1, [20, 10]).to(device=device)
        self.Cmq_model.load_state_dict(torch.load('./model/Cmq.pth', map_location=device))
        self.Cyp_model = MLP(1, 1, [20, 10]).to(device=device)
        self.Cyp_model.load_state_dict(torch.load('./model/Cyp.pth', map_location=device))
        self.Cyr_model = MLP(1, 1, [20, 10]).to(device=device)
        self.Cyr_model.load_state_dict(torch.load('./model/Cyr.pth', map_location=device))
        self.Cnr_model = MLP(1, 1, [20, 10]).to(device=device)
        self.Cnr_model.load_state_dict(torch.load('./model/Cnr.pth', map_location=device))
        self.Cnp_model = MLP(1, 1, [20, 10]).to(device=device)
        self.Cnp_model.load_state_dict(torch.load('./model/Cnp.pth', map_location=device))
        self.Clp_model = MLP(1, 1, [20, 10]).to(device=device)
        self.Clp_model.load_state_dict(torch.load('./model/Clp.pth', map_location=device))
        self.Clr_model = MLP(1, 1, [20, 10]).to(device=device)
        self.Clr_model.load_state_dict(torch.load('./model/Clr.pth', map_location=device))
        self.delta_Cx_lef_model = MLP(2, 1, [20, 10]).to(device=device)
        self.delta_Cx_lef_model.load_state_dict(torch.load('./model/delta_Cx_lef.pth', map_location=device))
        self.delta_Cz_lef_model = MLP(2, 1, [20, 10, 5]).to(device=device)
        self.delta_Cz_lef_model.load_state_dict(torch.load('./model/delta_Cz_lef.pth', map_location=device))
        self.delta_Cm_lef_model = MLP(2, 1, [20, 10, 5]).to(device=device)
        self.delta_Cm_lef_model.load_state_dict(torch.load('./model/delta_Cm_lef.pth', map_location=device))
        self.delta_Cy_lef_model = MLP(2, 1, [20, 10, 5]).to(device=device)
        self.delta_Cy_lef_model.load_state_dict(torch.load('./model/delta_Cy_lef.pth', map_location=device))
        self.delta_Cn_lef_model = MLP(2, 1, [20, 10, 5]).to(device=device)
        self.delta_Cn_lef_model.load_state_dict(torch.load('./model/delta_Cn_lef.pth', map_location=device))
        self.delta_Cl_lef_model = MLP(2, 1, [20, 10]).to(device=device)
        self.delta_Cl_lef_model.load_state_dict(torch.load('./model/delta_Cl_lef.pth', map_location=device))
        self.delta_Cxq_lef_model = MLP(1, 1, [20, 10]).to(device=device)
        self.delta_Cxq_lef_model.load_state_dict(torch.load('./model/delta_Cxq_lef.pth', map_location=device))
        self.delta_Cyr_lef_model = MLP(1, 1, [20, 10]).to(device=device)
        self.delta_Cyr_lef_model.load_state_dict(torch.load('./model/delta_Cyr_lef.pth', map_location=device))
        self.delta_Cyp_lef_model = MLP(1, 1, [20, 10, 5]).to(device=device)
        self.delta_Cyp_lef_model.load_state_dict(torch.load('./model/delta_Cyp_lef.pth', map_location=device))
        self.delta_Czq_lef_model = MLP(1, 1, [20, 10]).to(device=device)
        self.delta_Czq_lef_model.load_state_dict(torch.load('./model/delta_Czq_lef.pth', map_location=device))
        self.delta_Clr_lef_model = MLP(1, 1, [20, 10]).to(device=device)
        self.delta_Clr_lef_model.load_state_dict(torch.load('./model/delta_Clr_lef.pth', map_location=device))
        self.delta_Clp_lef_model = MLP(1, 1, [20, 10]).to(device=device)
        self.delta_Clp_lef_model.load_state_dict(torch.load('./model/delta_Clp_lef.pth', map_location=device))
        self.delta_Cmq_lef_model = MLP(1, 1, [20, 10]).to(device=device)
        self.delta_Cmq_lef_model.load_state_dict(torch.load('./model/delta_Cmq_lef.pth', map_location=device))
        self.delta_Cnr_lef_model = MLP(1, 1, [20, 10]).to(device=device)
        self.delta_Cnr_lef_model.load_state_dict(torch.load('./model/delta_Cnr_lef.pth', map_location=device))
        self.delta_Cnp_lef_model = MLP(1, 1, [20, 10]).to(device=device)
        self.delta_Cnp_lef_model.load_state_dict(torch.load('./model/delta_Cnp_lef.pth', map_location=device))
        self.delta_Cy_r30_model = MLP(2, 1, [20, 10, 5]).to(device=device)
        self.delta_Cy_r30_model.load_state_dict(torch.load('./model/delta_Cy_r30.pth', map_location=device))
        self.delta_Cn_r30_model = MLP(2, 1, [20, 10, 5]).to(device=device)
        self.delta_Cn_r30_model.load_state_dict(torch.load('./model/delta_Cn_r30.pth', map_location=device))
        self.delta_Cl_r30_model = MLP(2, 1, [20, 10, 5]).to(device=device)
        self.delta_Cl_r30_model.load_state_dict(torch.load('./model/delta_Cl_r30.pth', map_location=device))
        self.delta_Cy_a20_model = MLP(2, 1, [20, 10, 10]).to(device=device)
        self.delta_Cy_a20_model.load_state_dict(torch.load('./model/delta_Cy_a20.pth', map_location=device))
        self.delta_Cy_a20_lef_model = MLP(2, 1, [20, 20, 10]).to(device=device)
        self.delta_Cy_a20_lef_model.load_state_dict(torch.load('./model/delta_Cy_a20_lef.pth', map_location=device))
        self.delta_Cn_a20_model = MLP(2, 1, [20, 10, 5]).to(device=device)
        self.delta_Cn_a20_model.load_state_dict(torch.load('./model/delta_Cn_a20.pth', map_location=device))
        self.delta_Cn_a20_lef_model = MLP(2, 1, [20, 20, 10]).to(device=device)
        self.delta_Cn_a20_lef_model.load_state_dict(torch.load('./model/delta_Cn_a20_lef.pth', map_location=device))
        self.delta_Cl_a20_model = MLP(2, 1, [20, 10]).to(device=device)
        self.delta_Cl_a20_model.load_state_dict(torch.load('./model/delta_Cl_a20.pth', map_location=device))
        self.delta_Cl_a20_lef_model = MLP(2, 1, [20, 20, 10]).to(device=device)
        self.delta_Cl_a20_lef_model.load_state_dict(torch.load('./model/delta_Cl_a20_lef.pth', map_location=device))
        self.delta_Cnbeta_model = MLP(1, 1, [20, 10]).to(device=device)
        self.delta_Cnbeta_model.load_state_dict(torch.load('./model/delta_Cnbeta.pth', map_location=device))
        self.delta_Clbeta_model = MLP(1, 1, [20, 10]).to(device=device)
        self.delta_Clbeta_model.load_state_dict(torch.load('./model/delta_Clbeta.pth', map_location=device))
        self.delta_Cm_model = MLP(1, 1, [20, 10]).to(device=device)
        self.delta_Cm_model.load_state_dict(torch.load('./model/delta_Cm.pth', map_location=device))
        self.eta_el_model = MLP(1, 1, [20, 10]).to(device=device)
        self.eta_el_model.load_state_dict(torch.load('./model/eta_el.pth', map_location=device))

    def safe_read_dat(self, dat_name):
        try:
            if dat_name in HIFI_GLOBAL_TXT_CONTENT:
                return HIFI_GLOBAL_TXT_CONTENT.get(dat_name)

            data_path = path + r'/data/' + dat_name
            with open(data_path, 'r', encoding='utf-8') as file:
                content = file.read()
                content = content.strip()
                data_str = [value for value in content.split(' ') if value]
                data = list(map(float, data_str))
                data = torch.tensor(data, device=torch.device(device))
                HIFI_GLOBAL_TXT_CONTENT[dat_name] = data
                return data
        except OSError:
            print("Cannot find file {} in current directory".format(data_path))
            return []
        
    @torch.no_grad()
    def _Cx(self, alpha, beta, dele):
        name = self.data['name']
        index = list(name).index('Cx')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        el_mean = self.data['el_mean'][index]
        el_std = self.data['el_std'][index]
        dele = normalize(dele, el_mean, el_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        input = torch.hstack((input, dele.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.Cx_model.forward(input), mean, std)

    @torch.no_grad()
    def _Cz(self, alpha, beta, dele):
        name = self.data['name']
        index = list(name).index('Cz')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        el_mean = self.data['el_mean'][index]
        el_std = self.data['el_std'][index]
        dele = normalize(dele, el_mean, el_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        input = torch.hstack((input, dele.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.Cz_model.forward(input), mean, std)

    @torch.no_grad()
    def _Cm(self, alpha, beta, dele):
        name = self.data['name']
        index = list(name).index('Cm')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        el_mean = self.data['el_mean'][index]
        el_std = self.data['el_std'][index]
        dele = normalize(dele, el_mean, el_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        input = torch.hstack((input, dele.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.Cm_model.forward(input), mean, std)

    @torch.no_grad()
    def _Cy(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('Cy')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.Cy_model.forward(input), mean, std)

    @torch.no_grad()
    def _Cn(self, alpha, beta, dele):
        name = self.data['name']
        index = list(name).index('Cn')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        el_mean = self.data['el_mean'][index]
        el_std = self.data['el_std'][index]
        dele = normalize(dele, el_mean, el_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        input = torch.hstack((input, dele.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.Cn_model.forward(input), mean, std)

    @torch.no_grad()
    def _Cl(self, alpha, beta, dele):
        name = self.data['name']
        index = list(name).index('Cl')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        el_mean = self.data['el_mean'][index]
        el_std = self.data['el_std'][index]
        dele = normalize(dele, el_mean, el_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        input = torch.hstack((input, dele.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.Cl_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cx_lef(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('delta_Cx_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cx_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cz_lef(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('delta_Cz_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cz_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cm_lef(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('delta_Cm_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cm_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cy_lef(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('delta_Cy_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cy_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cn_lef(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('delta_Cn_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cn_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cl_lef(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('delta_Cl_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cl_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _CXq(self, alpha):
        name = self.data['name']
        index = list(name).index('Cxq')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.Cxq_model.forward(input), mean, std)

    @torch.no_grad()
    def _CZq(self, alpha):
        name = self.data['name']
        index = list(name).index('Czq')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.Czq_model.forward(input), mean, std)

    @torch.no_grad()
    def _CMq(self, alpha):
        name = self.data['name']
        index = list(name).index('Cmq')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.Cmq_model.forward(input), mean, std)

    @torch.no_grad()
    def _CYp(self, alpha):
        name = self.data['name']
        index = list(name).index('Cyp')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.Cyp_model.forward(input), mean, std)

    @torch.no_grad()
    def _CYr(self, alpha):
        name = self.data['name']
        index = list(name).index('Cyr')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.Cyr_model.forward(input), mean, std)

    @torch.no_grad()
    def _CNr(self, alpha):
        name = self.data['name']
        index = list(name).index('Cnr')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.Cnr_model.forward(input), mean, std)

    @torch.no_grad()
    def _CNp(self, alpha):
        name = self.data['name']
        index = list(name).index('Cnp')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.Cnp_model.forward(input), mean, std)

    @torch.no_grad()
    def _CLp(self, alpha):
        name = self.data['name']
        index = list(name).index('Clp')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.Clp_model.forward(input), mean, std)

    @torch.no_grad()
    def _CLr(self, alpha):
        name = self.data['name']
        index = list(name).index('Clr')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.Clr_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_CXq_lef(self, alpha):
        name = self.data['name']
        index = list(name).index('delta_Cxq_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cxq_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_CYr_lef(self, alpha):
        name = self.data['name']
        index = list(name).index('delta_Cyr_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cyr_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_CYp_lef(self, alpha):
        name = self.data['name']
        index = list(name).index('delta_Cyp_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cyp_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_CZq_lef(self, alpha):
        name = self.data['name']
        index = list(name).index('delta_Czq_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Czq_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_CLr_lef(self, alpha):
        name = self.data['name']
        index = list(name).index('delta_Clr_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Clr_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_CLp_lef(self, alpha):
        name = self.data['name']
        index = list(name).index('delta_Clp_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Clp_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_CMq_lef(self, alpha):
        name = self.data['name']
        index = list(name).index('delta_Cmq_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cmq_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_CNr_lef(self, alpha):
        name = self.data['name']
        index = list(name).index('delta_Cnr_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cnr_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_CNp_lef(self, alpha):
        name = self.data['name']
        index = list(name).index('delta_Cnp_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cnp_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cy_r30(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('delta_Cy_r30')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cy_r30_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cn_r30(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('delta_Cn_r30')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cn_r30_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cl_r30(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('delta_Cl_r30')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cl_r30_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cy_a20(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('delta_Cy_a20')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cy_a20_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cy_a20_lef(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('delta_Cy_a20_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cy_a20_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cn_a20(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('delta_Cn_a20')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cn_a20_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cn_a20_lef(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('delta_Cn_a20_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cn_a20_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cl_a20(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('delta_Cl_a20')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cl_a20_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cl_a20_lef(self, alpha, beta):
        name = self.data['name']
        index = list(name).index('delta_Cl_a20_lef')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        beta_mean = self.data['beta_mean'][index]
        beta_std = self.data['beta_std'][index]
        beta = normalize(beta, beta_mean, beta_std)
        input = torch.hstack((alpha.reshape(-1, 1), beta.reshape(-1, 1)))
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cl_a20_lef_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_CNbeta(self, alpha):
        name = self.data['name']
        index = list(name).index('delta_Cnbeta')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cnbeta_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_CLbeta(self, alpha):
        name = self.data['name']
        index = list(name).index('delta_Clbeta')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Clbeta_model.forward(input), mean, std)

    @torch.no_grad()
    def _delta_Cm(self, alpha):
        name = self.data['name']
        index = list(name).index('delta_Cm')
        alpha_mean = self.data['alpha_mean'][index]
        alpha_std = self.data['alpha_std'][index]
        alpha = normalize(alpha, alpha_mean, alpha_std)
        input = alpha.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.delta_Cm_model.forward(input), mean, std)

    @torch.no_grad()
    def _eta_el(self, el):
        name = self.data['name']
        index = list(name).index('eta_el')
        el_mean = self.data['el_mean'][index]
        el_std = self.data['el_std'][index]
        el = normalize(el, el_mean, el_std)
        input = el.reshape(-1, 1)
        mean = self.data['mean'][index]
        std = self.data['std'][index]
        return unnormalize(self.eta_el_model.forward(input), mean, std)

    def hifi_C(self, alpha, beta, el):
        return (
            self._Cx(alpha, beta, el),
            self._Cz(alpha, beta, el),
            self._Cm(alpha, beta, el),
            self._Cy(alpha, beta),
            self._Cn(alpha, beta, el),
            self._Cl(alpha, beta, el),
        )

    def hifi_damping(self, alpha):
        return (
            self._CXq(alpha),
            self._CYr(alpha),
            self._CYp(alpha),
            self._CZq(alpha),
            self._CLr(alpha),
            self._CLp(alpha),
            self._CMq(alpha),
            self._CNr(alpha),
            self._CNp(alpha),
        )

    def hifi_C_lef(self, alpha, beta):
        return (
            self._delta_Cx_lef(alpha, beta),
            self._delta_Cz_lef(alpha, beta),
            self._delta_Cm_lef(alpha, beta),
            self._delta_Cy_lef(alpha, beta),
            self._delta_Cn_lef(alpha, beta),
            self._delta_Cl_lef(alpha, beta),
        )

    def hifi_damping_lef(self, alpha):
        return (
            self._delta_CXq_lef(alpha),
            self._delta_CYr_lef(alpha),
            self._delta_CYp_lef(alpha),
            self._delta_CZq_lef(alpha),
            self._delta_CLr_lef(alpha),
            self._delta_CLp_lef(alpha),
            self._delta_CMq_lef(alpha),
            self._delta_CNr_lef(alpha),
            self._delta_CNp_lef(alpha),
        )

    def hifi_rudder(self, alpha, beta):
        return (
            self._delta_Cy_r30(alpha, beta),
            self._delta_Cn_r30(alpha, beta),
            self._delta_Cl_r30(alpha, beta),
        )

    def hifi_ailerons(self, alpha, beta):
        return (
            self._delta_Cy_a20(alpha, beta),
            self._delta_Cy_a20_lef(alpha, beta),
            self._delta_Cn_a20(alpha, beta),
            self._delta_Cn_a20_lef(alpha, beta),
            self._delta_Cl_a20(alpha, beta),
            self._delta_Cl_a20_lef(alpha, beta),
        )

    def hifi_other_coeffs(self, alpha, el):
        zero = torch.zeros_like(alpha)
        return (
            self._delta_CNbeta(alpha),
            self._delta_CLbeta(alpha),
            self._delta_Cm(alpha),
            self._eta_el(el),
            zero,
        )

In [4]:
def main(n, alpha, beta, el):
    hifi = hifi_F16()
    alpha_tensor = alpha * torch.ones(n, device=device)
    beta_tensor = beta * torch.ones(n, device=device)
    el_tensor = el * torch.ones(n, device=device)
    hifi._Cx(alpha_tensor, beta_tensor, el_tensor)

In [5]:
%timeit main(1000, 0.1, 0.2, 0.3)

47 ms ± 1.27 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [2]:
import torch
import torch.nn as nn


class F16Dynamics(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.hifi_F16 = hifi_F16(device=device)

    def compute_extended_state(self, x):
        return self.nlplant(x)

    def forward(self, t, x):
        es = self.compute_extended_state(x)
        return es
    
    def atmos(self, alt, vt):
        # 根据高度和速度计算动压、马赫数、静压
        rho0 = 2.377e-3
        tfac = 1 - .703e-5 * (alt)
        temp = 519.0 * tfac
        temp = (alt >= 35000.0) * 390 + (alt < 35000.0) * temp
        rho = rho0 * pow(tfac, 4.14)
        mach = (vt) / torch.sqrt(1.4 * 1716.3 * temp)
        qbar = .5 * rho * pow(vt, 2)
        ps = 1715.0 * rho * temp

        ps = (ps == 0) * 1715 + (ps != 0) * ps

        return (mach, qbar, ps)
    
    def nlplant(self, x):
        """
        model state(dim 12):
            0. ego_north_position      (unit: feet)
            1. ego_east_position       (unit: feet)
            2. ego_altitude            (unit: feet)
            3. ego_roll                (unit: rad)
            4. ego_pitch               (unit: rad)
            5. ego_yaw                 (unit: rad)
            6. ego_vt                  (unit: feet/s)
            7. ego_alpha               (unit: rad)
            8. ego_beta                (unit: rad)
            9. ego_P                   (unit: rad/s)
            10. ego_Q                  (unit: rad/s)
            11. ego_R                  (unit: rad/s)

        model control(dim 5)
            0. ego_T                  (unit: lbf)
            1. ego_el                 (unit: deg)
            2. ego_ail                (unit: deg)
            3. ego_rud                (unit: deg)
            4. ego_lef                (unit: deg)
        """
        xdot = torch.zeros_like(x)
        g = 32.17
        m = 636.94
        B = 30.0
        S = 300.0
        cbar = 11.32
        xcgr = 0.35
        xcg = 0.30
        Heng = 0.0
        pi = torch.pi

        Jy = 55814.0
        Jxz = 982.0
        Jz = 63100.0
        Jx = 9496.0

        r2d = 180.0 / pi

        # States
        alt = x[:, 2]
        phi = x[:, 3]
        theta = x[:, 4]
        psi = x[:, 5]

        vt = x[:, 6]
        alpha = x[:, 7] * r2d
        beta = x[:, 8] * r2d
        P = x[:, 9]
        Q = x[:, 10]
        R = x[:, 11]

        sa = torch.sin(x[:, 7])
        ca = torch.cos(x[:, 7])
        sb = torch.sin(x[:, 8])
        cb = torch.cos(x[:, 8])

        st = torch.sin(theta)
        ct = torch.cos(theta)
        tt = torch.tan(theta)
        sphi = torch.sin(phi)
        cphi = torch.cos(phi)
        spsi = torch.sin(psi)
        cpsi = torch.cos(psi)

        vt = (vt <= 0.01) * 0.01 + (vt > 0.01) * vt

        # Control inputs

        T = x[:, 12]
        el = x[:, 13]
        ail = x[:, 14]
        rud = x[:, 15]
        lef = x[:, 16]

        dail = ail / 21.5
        drud = rud / 30.0
        dlef = (1 - lef / 25.0)

        # Atmospheric effects
        # sets dynamic pressure and mach number

        temp = self.atmos(alt, vt)
        mach = temp[0]
        qbar = temp[1] # dynamic pressure
        ps = temp[2]

        # Dynamics
        # Navigation Equations

        U = vt * ca * cb
        V = vt * sb
        W = vt * sa * cb

        xdot[:, 0] = U * (ct * cpsi) + V * (sphi * cpsi * st - cphi * spsi) + W * (cphi * st * cpsi + sphi * spsi)
        xdot[:, 1] = U * (ct * spsi) + V * (sphi * spsi * st + cphi * cpsi) + W * (cphi * st * spsi - sphi * cpsi)
        xdot[:, 2] = U * st - V * (sphi * ct) - W * (cphi * ct)
        xdot[:, 3] = P + tt * (Q * sphi + R * cphi)
        xdot[:, 4] = Q * cphi - R * sphi
        xdot[:, 5] = (Q * sphi + R * cphi) / ct

        temp = self.hifi_F16.hifi_C(alpha, beta, el)
        Cx = temp[0]
        Cz = temp[1]
        Cm = temp[2]
        Cy = temp[3]
        Cn = temp[4]
        Cl = temp[5]

        temp = self.hifi_F16.hifi_damping(alpha)
        Cxq = temp[0]
        Cyr = temp[1]
        Cyp = temp[2]
        Czq = temp[3]
        Clr = temp[4]
        Clp = temp[5]
        Cmq = temp[6]
        Cnr = temp[7]
        Cnp = temp[8]

        temp = self.hifi_F16.hifi_C_lef(alpha, beta)
        delta_Cx_lef = temp[0]
        delta_Cz_lef = temp[1]
        delta_Cm_lef = temp[2]
        delta_Cy_lef = temp[3]
        delta_Cn_lef = temp[4]
        delta_Cl_lef = temp[5]

        temp = self.hifi_F16.hifi_damping_lef(alpha)
        delta_Cxq_lef = temp[0]
        delta_Cyr_lef = temp[1]
        delta_Cyp_lef = temp[2]
        delta_Clr_lef = temp[4]
        delta_Clp_lef = temp[5]
        delta_Cmq_lef = temp[6]
        delta_Cnr_lef = temp[7]
        delta_Cnp_lef = temp[8]

        temp = self.hifi_F16.hifi_rudder(alpha, beta)
        delta_Cy_r30 = temp[0]
        delta_Cn_r30 = temp[1]
        delta_Cl_r30 = temp[2]

        temp = self.hifi_F16.hifi_ailerons(alpha, beta)
        delta_Cy_a20 = temp[0]
        delta_Cy_a20_lef = temp[1]
        delta_Cn_a20 = temp[2]
        delta_Cn_a20_lef = temp[3]
        delta_Cl_a20 = temp[4]
        delta_Cl_a20_lef = temp[5]

        temp = self.hifi_F16.hifi_other_coeffs(alpha, el)
        delta_Cnbeta = temp[0]
        delta_Clbeta = temp[1]
        delta_Cm = temp[2]
        eta_el = temp[3]
        delta_Cm_ds = temp[4]
        
        dXdQ = (cbar / (2 * vt)) * (Cxq + delta_Cxq_lef * dlef)
        Cx_tot = Cx + delta_Cx_lef * dlef + dXdQ * Q
        dZdQ = (cbar / (2 * vt)) * (Czq + delta_Cz_lef * dlef)
        Cz_tot = Cz + delta_Cz_lef * dlef + dZdQ * Q
        dMdQ = (cbar / (2 * vt)) * (Cmq + delta_Cmq_lef * dlef)
        Cm_tot = Cm * eta_el + Cz_tot * (xcgr - xcg) + delta_Cm_lef * dlef + dMdQ * Q + delta_Cm + delta_Cm_ds
        dYdail = delta_Cy_a20 + delta_Cy_a20_lef * dlef
        dYdR = (B / (2 * vt)) * (Cyr + delta_Cyr_lef * dlef)
        dYdP = (B / (2 * vt)) * (Cyp + delta_Cyp_lef * dlef)
        Cy_tot = Cy + delta_Cy_lef * dlef + dYdail * dail + delta_Cy_r30 * drud + dYdR * R + dYdP * P
        dNdail = delta_Cn_a20 + delta_Cn_a20_lef * dlef
        dNdR = (B / (2 * vt)) * (Cnr + delta_Cnr_lef * dlef)
        dNdP = (B / (2 * vt)) * (Cnp + delta_Cnp_lef * dlef)
        Cn_tot = Cn + delta_Cn_lef * dlef - Cy_tot * (xcgr - xcg) * (cbar / B) + dNdail * dail + delta_Cn_r30 * drud + dNdR * R + dNdP * P + delta_Cnbeta * beta
        dLdail = delta_Cl_a20 + delta_Cl_a20_lef * dlef
        dLdR = (B / (2 * vt)) * (Clr + delta_Clr_lef * dlef)
        dLdP = (B / (2 * vt)) * (Clp + delta_Clp_lef * dlef)
        Cl_tot = Cl + delta_Cl_lef * dlef + dLdail * dail + delta_Cl_r30 * drud + dLdR * R + dLdP * P + delta_Clbeta * beta
        Udot = R * V - Q * W - g * st + qbar * S * Cx_tot / m + T / m
        Vdot = P * W - R * U + g * ct * sphi + qbar * S * Cy_tot / m
        Wdot = Q * U - P * V + g * ct * cphi + qbar * S * Cz_tot / m
        xdot[:, 6] = (U * Udot + V * Vdot + W * Wdot) / vt
        xdot[:, 7] = (U * Wdot - W * Udot) / (U * U + W * W)
        xdot[:, 8] = (Vdot * vt - V * xdot[:, 6]) / (vt * vt * cb)
        L_tot = Cl_tot * qbar * S * B
        M_tot = Cm_tot * qbar * S * cbar
        N_tot = Cn_tot * qbar * S * B
        denom = Jx * Jz - Jxz * Jxz
        xdot[:, 9] = (Jz * L_tot + Jxz * N_tot - (Jz * (Jz - Jy) + Jxz * Jxz) * Q * R + Jxz * (Jx - Jy + Jz) * P * Q + Jxz * Q * Heng) / denom
        xdot[:, 10] = (M_tot + (Jz - Jx) * P * R - Jxz * (P * P - R * R) - R * Heng) / Jy
        xdot[:, 11] = (Jx * N_tot + Jxz * L_tot + (Jx * (Jx - Jy) + Jxz * Jxz) * P * Q - Jxz * (Jx - Jy + Jz) * Q * R + Jx * Q * Heng) / denom

        return xdot

### 4. Build your model
Build your own model, more details about your model's interface see in `../envs/models/model_base.py`

In [3]:
import os
import sys
import torch
from torchdiffeq import odeint_adjoint as odeint
sys.path.append(os.path.dirname(os.path.abspath('.')))
from envs.models.model_base import BaseModel


class F16Model(BaseModel):
    def __init__(self, config, n, device, random_seed):
        super().__init__(config, n, device, random_seed)
        self.num_states = getattr(self.config, 'num_states', 12)
        self.num_controls = getattr(self.config, 'num_controls', 5)
        self.dt = getattr(self.config, 'dt', 0.02)
        self.solver = getattr(self.config, 'solver', 'euler')
        self.airspeed = getattr(self.config, 'airspeed', 0)

        self.s = torch.zeros((self.n, self.num_states), device=self.device)  # state
        self.recent_s = torch.zeros((self.n, self.num_states), device=self.device)  # recent state
        self.u = torch.zeros((self.n, self.num_controls), device=self.device) # control
        self.recent_u = torch.zeros((self.n, self.num_controls), device=self.device)  # recent control

        # init parameters
        self.max_altitude = getattr(self.config, 'max_altitude', 20000)
        self.min_altitude = getattr(self.config, 'min_altitude', 19000)
        self.max_vt = getattr(self.config, 'max_vt', 1200)
        self.min_vt = getattr(self.config, 'min_vt', 1000)
        # self.init_state = self.config.init_state

        self.dynamics = F16Dynamics(device)

    def reset(self, env):
        done = env.is_done.bool()
        bad_done = env.bad_done.bool()
        exceed_time_limit = env.exceed_time_limit.bool()
        reset = (done | bad_done) | exceed_time_limit
        size = torch.sum(reset)
        self.s[reset, :] = torch.zeros((size, self.num_states), device=self.device)  # state
        self.u[reset, :] = torch.zeros((size, self.num_controls), device=self.device)
        self.s[reset, 2] = torch.rand_like(self.s[reset, 2]) * (self.max_altitude - self.min_altitude) + self.min_altitude
        self.s[reset, 6] = torch.rand_like(self.s[reset, 6]) * (self.max_vt - self.min_vt) + self.min_vt
        # self.u[reset, 0] = self.init_state['init_T']
        self.recent_s[reset] = self.s[reset]
        self.recent_u[reset] = self.u[reset]

    def get_extended_state(self):
        x = torch.hstack((self.s, self.u))
        return self.dynamics.nlplant(x)
    
    def update(self, action):
        action = torch.clamp(action, -1, 1)
        T = 0.9 * self.u[:, 0].reshape(-1, 1) + 0.1 * action[:, 0].reshape(-1, 1) * 0.225 * 76300 / 0.3048
        el = 0.9 * self.u[:, 1].reshape(-1, 1) + 0.1 * action[:, 1].reshape(-1, 1) * 45
        ail = 0.9 * self.u[:, 2].reshape(-1, 1) + 0.1 * action[:, 2].reshape(-1, 1) * 45
        rud = 0.9 * self.u[:, 3].reshape(-1, 1) + 0.1 * action[:, 3].reshape(-1, 1) * 45
        lef = torch.zeros((self.n, 1), device=self.device)
        self.recent_u = self.u
        self.u = torch.hstack((T, el))
        self.u = torch.hstack((self.u, ail))
        self.u = torch.hstack((self.u, rud))
        self.u = torch.hstack((self.u, lef))
        self.recent_s = self.s
        self.s = odeint(self.dynamics,
                        torch.hstack((self.s, self.u)),
                        torch.tensor([0., self.dt], device=self.device),
                        method=self.solver)[1, :, :self.num_states]
    
    def get_state(self):
        return self.s
    
    def get_control(self):
        return self.u
    
    def get_position(self):
        return self.s[:, 0], self.s[:, 1], self.s[:, 2]
    
    def get_ground_speed(self):
        es = self.get_extended_state()
        return es[:, 0], es[:, 1]
    
    def get_climb_rate(self):
        es = self.get_extended_state()
        return es[:, 2]
    
    def get_posture(self):
        return self.s[:, 3], self.s[:, 4], self.s[:, 5]
    
    def get_euler_angular_velocity(self):
        es = self.get_extended_state()
        return es[:, 3], es[:, 4], es[:, 5]
    
    def get_vt(self):
        return self.s[:, 6]
    
    def get_TAS(self):
        return self.s[:, 6] + self.airspeed * torch.ones_like(self.s[:, 6])
    
    def get_EAS(self):
        TAS = self.get_TAS()
        EAS2TAS = self.get_EAS2TAS()
        EAS = TAS / EAS2TAS
        return EAS
    
    def get_AOA(self):
        return self.s[:, 7]
    
    def get_AOS(self):
        return self.s[:, 8]
    
    def get_angular_velocity(self):
        return self.s[:, 9], self.s[:, 10], self.s[:, 11]
    
    def get_thrust(self):
        return self.u[:, 0]
    
    def get_control_surface(self):
        return self.u[:, 1], self.u[:, 2], self.u[:, 3], self.u[:, 4]

    
    def get_velocity(self):
        # 根据飞行状态计算三轴速度
        sina = torch.sin(self.s[:, 7])
        cosa = torch.cos(self.s[:, 7])
        sinb = torch.sin(self.s[:, 8])
        cosb = torch.cos(self.s[:, 8])
        vel_u = self.s[:, 6] * cosb * cosa # x轴速度
        vel_v = self.s[:, 6] * sinb # y轴速度
        vel_w = self.s[:, 6] * cosb * sina # z轴速度
        return vel_u, vel_v, vel_w
    
    def get_acceleration(self):
        # 根据飞行状态计算三轴加速度
        xdot = self.get_extended_state()
        sina = torch.sin(self.s[:, 7])
        cosa = torch.cos(self.s[:, 7])
        sinb = torch.sin(self.s[:, 8])
        cosb = torch.cos(self.s[:, 8])
        vel_u = self.s[:, 6] * cosb * cosa # x轴速度
        vel_v = self.s[:, 6] * sinb # y轴速度
        vel_w = self.s[:, 6] * cosb * sina # z轴速度
        u_dot = cosb * cosa * xdot[:, 6] - self.s[:, 6] * sinb * cosa * xdot[:, 8] - self.s[:, 6] * cosb * sina * xdot[:, 7]
        v_dot = sinb * xdot[:, 6] + self.s[:, 6] * cosb * xdot[:, 8]
        w_dot = cosb * sina * xdot[:, 6] - self.s[:, 6] * sinb * sina * xdot[:, 8] + self.s[:, 6] * cosb * cosa * xdot[:, 7]
        ax = u_dot + self.s[:, 10] * vel_w - self.s[:, 11] * vel_v
        ay = v_dot + self.s[:, 11] * vel_u - self.s[:, 9] * vel_w
        az = w_dot + self.s[:, 9] * vel_v - self.s[:, 10] * vel_u
        return ax, ay, az
    
    def get_G(self):
        # 根据飞行状态计算过载
        nx_cg, ny_cg, nz_cg = self.get_accels()
        G = torch.sqrt(nx_cg ** 2 + ny_cg ** 2 + nz_cg ** 2)
        return G
    
    def get_EAS2TAS(self):
        # 根据高度计算EAS2TAS
        alt = self.s[:, 2]
        tfac = 1 - .703e-5 * (alt)
        eas2tas = 1 / torch.pow(tfac, 4.14)
        eas2tas = torch.sqrt(eas2tas)
        return eas2tas
    
    def get_accels(self):
        # 根据飞行状态计算三轴过载
        grav = 32.174
        xdot = self.get_extended_state()
        sina = torch.sin(self.s[:, 7])
        cosa = torch.cos(self.s[:, 7])
        sinb = torch.sin(self.s[:, 8])
        cosb = torch.cos(self.s[:, 8])
        vel_u = self.s[:, 6] * cosb * cosa
        vel_v = self.s[:, 6] * sinb
        vel_w = self.s[:, 6] * cosb * sina
        u_dot = cosb * cosa * xdot[:, 6] - self.s[:, 6] * sinb * cosa * xdot[:, 8] - self.s[:, 6] * cosb * sina * xdot[:, 7]
        v_dot = sinb * xdot[:, 6] + self.s[:, 6] * cosb * xdot[:, 8]
        w_dot = cosb * sina * xdot[:, 6] - self.s[:, 6] * sinb * sina * xdot[:, 8] + self.s[:, 6] * cosb * cosa * xdot[:, 7]
        nx_cg = 1.0 / grav * (u_dot + self.s[:, 10] * vel_w - self.s[:, 11] * vel_v) + torch.sin(self.s[:, 4])
        ny_cg = 1.0 / grav * (v_dot + self.s[:, 11] * vel_u - self.s[:, 9] * vel_w) - torch.cos(self.s[:, 4]) * torch.sin(self.s[:, 3])
        nz_cg = -1.0 / grav * (w_dot + self.s[:, 9] * vel_v - self.s[:, 10] * vel_u) + torch.cos(self.s[:, 4]) * torch.cos(self.s[:, 3])
        return nx_cg, ny_cg, nz_cg

    def get_atmos(self):
        # 根据高度和速度计算动压、马赫数、静压
        alt = self.s[:, 2]
        vt = self.s[:, 6]
        rho0 = 2.377e-3
        tfac = 1 - .703e-5 * (alt)
        temp = 519.0 * tfac
        temp = (alt >= 35000.0) * 390 + (alt < 35000.0) * temp
        rho = rho0 * pow(tfac, 4.14)
        mach = (vt) / torch.sqrt(1.4 * 1716.3 * temp)
        qbar = .5 * rho * pow(vt, 2)
        ps = 1715.0 * rho * temp

        ps = (ps == 0) * 1715 + (ps != 0) * ps

        return (mach, qbar, ps)

### 5. Test your model

In [4]:
from tqdm import tqdm


model = F16Model(config=None, n=10000, device=torch.device('cpu'), random_seed=42)
for i in tqdm(range(1000)):
    action = torch.zeros((10000, 5), device=torch.device('cpu'))
    model.update(action)

100%|██████████| 1000/1000 [00:54<00:00, 18.47it/s]
